In [ ]:
import pandas as pd


df = pd.read_csv(r'C:\Users\z0047u3y\Documents\Cristi work\data\dataset_final.csv', delimiter=';')

# Generate 20 lags for all columns
for col in df.columns:
    if col != 'label':  # Don't create lags for the label column
        for lag in range(1, 21):
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)

# Drop rows with NaN values created by lagging
df = df.dropna()

print(f"Dataset shape after adding lags: {df.shape}")
print(df.head())

Dataset shape after adding lags: (538307, 21)
           ax,ay,az,gx,gy,gz,temp,label,pwm_speed  \
20   0.11,1.24,-10.7,-0.09,0.02,-0.01,24.09,2,160   
21  0.11,1.16,-10.68,-0.08,0.03,-0.01,24.09,2,160   
22  0.17,1.18,-10.58,-0.09,0.02,-0.01,24.11,2,160   
23    0.15,1.14,-10.7,-0.09,0.02,-0.01,24.1,2,160   
24  0.16,1.21,-10.76,-0.07,0.02,-0.01,24.12,2,160   

     ax,ay,az,gx,gy,gz,temp,label,pwm_speed_lag_1  \
20   0.06,1.1,-10.56,-0.08,0.01,-0.01,24.12,2,160   
21   0.11,1.24,-10.7,-0.09,0.02,-0.01,24.09,2,160   
22  0.11,1.16,-10.68,-0.08,0.03,-0.01,24.09,2,160   
23  0.17,1.18,-10.58,-0.09,0.02,-0.01,24.11,2,160   
24    0.15,1.14,-10.7,-0.09,0.02,-0.01,24.1,2,160   

     ax,ay,az,gx,gy,gz,temp,label,pwm_speed_lag_2  \
20   0.15,1.24,-10.67,-0.07,0.03,-0.01,24.1,2,160   
21   0.06,1.1,-10.56,-0.08,0.01,-0.01,24.12,2,160   
22   0.11,1.24,-10.7,-0.09,0.02,-0.01,24.09,2,160   
23  0.11,1.16,-10.68,-0.08,0.03,-0.01,24.09,2,160   
24  0.17,1.18,-10.58,-0.09,0.02,-0.01,24.11,2,160  

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Separate features and target
X = df.drop('label', axis=1)
y = df['label']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

KeyboardInterrupt: 

In [7]:
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("WARNING: CUDA not available. Running on CPU.")
    print("Install GPU version: pip install torch --index-url https://download.pytorch.org/whl/cu121")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# ============================================================
# STEP 1: Define parameters
# ============================================================

X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("=" * 60)
print("STEP 1: Parameters")
print("=" * 60)
print(f"  X shape (flat, all features + lags): {X.shape}")
print(f"  Number of input features: {X.shape[1]}")
print(f"  Unique labels: {list(y.unique())}")
print(f"  First 8 columns (original sensors): {list(X.columns[:8])}")
print(f"  Example lag columns: {list(X.columns[8:12])}...")
print(f"\n  No reshaping needed — we feed the entire flat row directly to the network.")

# ============================================================
# STEP 2: Normalize (standardize) the features
# ============================================================
# StandardScaler makes each feature have mean=0, std=1.
# We fit on training data ONLY to prevent data leakage.

print("\n" + "=" * 60)
print("STEP 2: Feature normalization (StandardScaler)")
print("=" * 60)

print(f"  Before scaling — sample feature stats from training set:")
sample_cols = list(X.columns[:8])
for col in sample_cols:
    vals = X_train[col]
    print(f"    {col:12s}: mean={vals.mean():10.2f}, std={vals.std():10.2f}, "
          f"min={vals.min():10.2f}, max={vals.max():10.2f}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print(f"\n  After scaling — same features (should be ~mean=0, std=1):")
for i, col in enumerate(sample_cols):
    print(f"    {col:12s}: mean={X_train_scaled[:, i].mean():7.4f}, std={X_train_scaled[:, i].std():7.4f}")

print(f"\n  X_train_scaled shape: {X_train_scaled.shape}")
print(f"  X_test_scaled shape:  {X_test_scaled.shape}")

# ============================================================
# STEP 3: Encode labels automatically with LabelEncoder
# ============================================================
# LabelEncoder converts string labels to consecutive integers (0, 1, 2, ...)
# We use it here only because PyTorch requires integer tensors for CrossEntropyLoss.
# The human-readable names are recovered at the end for the classification report.

print("\n" + "=" * 60)
print("STEP 3: Label encoding (LabelEncoder — automatic)")
print("=" * 60)

le = LabelEncoder()
le.fit(y)  # Fit on ALL labels so train & test share the same encoding
num_classes = len(le.classes_)

y_train_encoded = le.transform(y_train).astype(np.int64)
y_test_encoded = le.transform(y_test).astype(np.int64)

print(f"  LabelEncoder classes: {list(le.classes_)}")
print(f"  Automatically assigned: {dict(zip(le.classes_, range(num_classes)))}")
print(f"  Number of classes: {num_classes}")
print(f"  Original labels (first 10): {list(y_train.values[:10])}")
print(f"  Encoded labels  (first 10): {list(y_train_encoded[:10])}")
print(f"\n  Train label distribution:")
unique, counts = np.unique(y_train_encoded, return_counts=True)
# for u, c in zip(unique, counts):
#     print(f"    {le.classes_[u]:15s} (class {u}): {c:6d} samples ({100*c/len(y_train_encoded):.1f}%)")

# ============================================================
# STEP 4: Convert to PyTorch tensors and create DataLoaders
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Convert to PyTorch tensors & create DataLoaders")
print("=" * 60)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

print(f"  X_train_tensor: {X_train_tensor.shape}, dtype={X_train_tensor.dtype}")
print(f"  y_train_tensor: {y_train_tensor.shape}, dtype={y_train_tensor.dtype}")
print(f"  X_test_tensor:  {X_test_tensor.shape}, dtype={X_test_tensor.dtype}")
print(f"  y_test_tensor:  {y_test_tensor.shape}, dtype={y_test_tensor.dtype}")

# Split training into train/validation (80/20)
val_size = int(0.2 * len(X_train_tensor))
train_size = len(X_train_tensor) - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    TensorDataset(X_train_tensor, y_train_tensor), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

print(f"\n  DataLoader splits:")
print(f"    Training:   {train_size} samples → {len(train_loader)} batches of {batch_size}")
print(f"    Validation: {val_size} samples → {len(val_loader)} batches of {batch_size}")
print(f"    Test:       {len(X_test_tensor)} samples → {len(test_loader)} batches of {batch_size}")

# ============================================================
# STEP 5: Define the MLP model
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Model architecture (MLP — fully connected)")
print("=" * 60)

input_size = X_train_scaled.shape[1]

class MLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),    # Input → 256 neurons
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),           # 256 → 128 neurons
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),            # 128 → 64 neurons
            nn.ReLU(),
            nn.Linear(64, num_classes)     # 64 → 5 output classes (logits)
        )
    
    def forward(self, x):
        return self.network(x)

# Set device (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Using device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model_mlp = MLP(input_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)

print(f"\n  Model summary:")
print(f"  {model_mlp}")
total_params = sum(p.numel() for p in model_mlp.parameters())
trainable_params = sum(p.numel() for p in model_mlp.parameters() if p.requires_grad)
print(f"\n  Input size: {input_size} (all features + lags in a single flat vector)")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Verify a dummy forward pass
dummy = torch.randn(1, input_size).to(device)
dummy_out = model_mlp(dummy)
print(f"\n  Dummy forward pass:")
print(f"    Input:  {dummy.shape} — (1 sample, {input_size} features)")
print(f"    Output: {dummy_out.shape} — (1 sample, {num_classes} class logits)")

# ============================================================
# STEP 6: Training loop
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Training")
print("=" * 60)

epochs = 20
for epoch in range(epochs):
    model_mlp.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model_mlp(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += batch_y.size(0)
        train_correct += (predicted == batch_y).sum().item()
    
    # Validation
    model_mlp.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model_mlp(batch_X)
            loss = criterion(outputs, batch_y)
            
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    print(f"  Epoch {epoch+1:2d}/{epochs} | "
          f"Train Loss: {train_loss/train_total:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss/val_total:.4f} | Val Acc: {val_acc:.4f}")

# ============================================================
# STEP 7: Evaluate on test set
# ============================================================
print("\n" + "=" * 60)
print("STEP 7: Test set evaluation")
print("=" * 60)

model_mlp.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model_mlp(batch_X)
        loss = criterion(outputs, batch_y)
        
        test_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        test_total += batch_y.size(0)
        test_correct += (predicted == batch_y).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(batch_y.cpu().numpy())

test_accuracy = test_correct / test_total
print(f"\n  Test Loss: {test_loss/test_total:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

# Map integer predictions back to human-readable names for the report
y_pred_classes = le.inverse_transform(all_preds)
y_target_classes = le.inverse_transform(all_targets)

print(f"\n  Classification Report:")
print(classification_report(y_target_classes, y_pred_classes))

In [ ]:
test_accuracy = test_correct / test_total
print(f"\n  Test Loss: {test_loss/test_total:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

# Map integer predictions back to human-readable names for the report
y_pred_classes = le.inverse_transform(all_preds)
y_target_classes = le.inverse_transform(all_targets)

print(f"\n  Classification Report:")
print(classification_report(y_target_classes, y_pred_classes))

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# ============================================================
# STEP 1: Define parameters
# ============================================================

X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("=" * 60)
print("STEP 1: Parameters")
print("=" * 60)
print(f"  X shape (flat, all features + lags): {X.shape}")
print(f"  Number of input features: {X.shape[1]}")
print(f"  Unique labels: {list(y.unique())}")
print(f"  First 8 columns (original sensors): {list(X.columns[:8])}")
print(f"  Example lag columns: {list(X.columns[8:12])}...")
print(f"\n  No reshaping needed — we feed the entire flat row directly to the network.")

# Check GPU availability
print(f"\n  TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"  GPU available: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("  WARNING: No GPU detected. Running on CPU.")

# ============================================================
# STEP 2: Normalize (standardize) the features
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Feature normalization (StandardScaler)")
print("=" * 60)

print(f"  Before scaling — sample feature stats from training set:")
sample_cols = list(X.columns[:8])
for col in sample_cols:
    vals = X_train[col]
    print(f"    {col:12s}: mean={vals.mean():10.2f}, std={vals.std():10.2f}, "
          f"min={vals.min():10.2f}, max={vals.max():10.2f}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print(f"\n  After scaling — same features (should be ~mean=0, std=1):")
for i, col in enumerate(sample_cols):
    print(f"    {col:12s}: mean={X_train_scaled[:, i].mean():7.4f}, std={X_train_scaled[:, i].std():7.4f}")

print(f"\n  X_train_scaled shape: {X_train_scaled.shape}")
print(f"  X_test_scaled shape:  {X_test_scaled.shape}")

# ============================================================
# STEP 3: Encode labels automatically with LabelEncoder
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Label encoding (LabelEncoder — automatic)")
print("=" * 60)

le = LabelEncoder()
le.fit(y)
num_classes = len(le.classes_)

y_train_encoded = le.transform(y_train).astype(np.int64)
y_test_encoded = le.transform(y_test).astype(np.int64)

print(f"  LabelEncoder classes: {list(le.classes_)}")
print(f"  Automatically assigned: {dict(zip(le.classes_, range(num_classes)))}")
print(f"  Number of classes: {num_classes}")
print(f"  Original labels (first 10): {list(y_train.values[:10])}")
print(f"  Encoded labels  (first 10): {list(y_train_encoded[:10])}")
print(f"\n  Train label distribution:")
unique, counts = np.unique(y_train_encoded, return_counts=True)

# ============================================================
# STEP 4: Build the MLP model
# ============================================================
# Same architecture as the PyTorch version:
# Input → 256 → 128 → 64 → num_classes

print("\n" + "=" * 60)
print("STEP 4: Model architecture (MLP — fully connected)")
print("=" * 60)

input_size = X_train_scaled.shape[1]

model_mlp_tf = keras.Sequential([
    layers.Input(shape=(input_size,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model_mlp_tf.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"  Input size: {input_size} (all features + lags in a single flat vector)")
model_mlp_tf.summary()

# ============================================================
# STEP 5: Training
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Training")
print("=" * 60)

history = model_mlp_tf.fit(
    X_train_scaled, y_train_encoded,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# ============================================================
# STEP 6: Evaluate on test set
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Test set evaluation")
print("=" * 60)

test_loss, test_accuracy = model_mlp_tf.evaluate(X_test_scaled, y_test_encoded, verbose=0)
print(f"\n  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

# Get predictions
y_pred_probs = model_mlp_tf.predict(X_test_scaled, verbose=0)
y_pred_encoded = np.argmax(y_pred_probs, axis=1)

# Map integer predictions back to human-readable names for the report
y_pred_classes = le.inverse_transform(y_pred_encoded)
y_target_classes = le.inverse_transform(y_test_encoded)

print(f"\n  Classification Report:")
print(classification_report(y_target_classes, y_pred_classes))

STEP 1: Parameters
  X shape (flat, all features + lags): (538307, 168)
  Number of input features: 168
  Unique labels: [np.int64(2), np.int64(4), np.int64(3), np.int64(1), np.int64(0)]
  First 8 columns (original sensors): ['ax', 'ay', 'az', 'gx', 'gy', 'gz', 'temp', 'pwm_speed']
  Example lag columns: ['ax_lag_1', 'ax_lag_2', 'ax_lag_3', 'ax_lag_4']...

  No reshaping needed — we feed the entire flat row directly to the network.

  TensorFlow version: 2.20.0

STEP 2: Feature normalization (StandardScaler)
  Before scaling — sample feature stats from training set:
    ax          : mean=      0.16, std=      0.32, min=     -2.72, max=     41.00
    ay          : mean=      1.26, std=      0.40, min=     -1.32, max=      4.89
    az          : mean=    -10.62, std=      0.26, min=    -13.42, max=      0.00
    gx          : mean=     -0.08, std=      0.02, min=     -0.28, max=      0.13
    gy          : mean=      0.02, std=      0.03, min=     -0.27, max=      0.68
    gz          :

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        43,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 84,741 (331.02 KB)

 Trainable params: 84,741 (331.02 KB)

 Non-trainable params: 0 (0.00 B)


STEP 5: Training
Epoch 1/20
2692/2692 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9907 - loss: 0.0285 - val_accuracy: 0.9995 - val_loss: 0.0026
Epoch 2/20
2692/2692 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9983 - loss: 0.0059 - val_accuracy: 0.9996 - val_loss: 0.0020
Epoch 3/20
2692/2692 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9989 - loss: 0.0044 - val_accuracy: 0.9998 - val_loss: 0.0012
Epoch 4/20
2692/2692 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9992 - loss: 0.0032 - val_accuracy: 0.9998 - val_loss: 9.2397e-04
Epoch 5/20
2692/2692 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9992 - loss: 0.0032 - val_accuracy: 0.9998 - val_loss: 0.0012
Epoch 6/20
2692/2692 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9993 - loss: 0.0025 - val_accuracy: 0.9998 - val_loss: 8.6281e-04
Epoch 7/20
2692/2692 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9995 - loss: 0.0025 - val_accuracy: 0.9998 - val_loss: 0.0013
Epoch 8/20
2692/2692 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accura